# ETL 01 — Extração das séries macroeconômicas (BACEN/SGS)

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Fonte:** Sistema Gerenciador de Séries Temporais (SGS) do Banco Central do Brasil — dados abertos públicos.

**Período:** 01/01/2019 a 31/12/2023 (período oficial da Linha 4, conforme orientação do professor).

**Saída:** CSVs brutos em `data/raw/` (não versionados no Git; reproduzíveis executando este notebook).

In [1]:
from pathlib import Path
import time
import requests
import pandas as pd

# Raiz do projeto: funciona rodando de dentro de notebooks/ ou da raiz
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT)

Raiz do projeto: c:\Users\waldinei.rosa\Projetos\Projeto_Triade_Financeira_Waldinei


In [2]:
SERIES_SGS = {
    432: "selic_meta",  # Meta Selic definida pelo Copom (% a.a.) - diária
    433: "ipca_mensal",  # IPCA - variação mensal (%)
    13522: "ipca_12m",  # IPCA acumulado em 12 meses (%)
    1: "ptax_venda",  # Dólar comercial PTAX - venda (R$) - dias úteis
}

DATA_INICIAL = "01/01/2019"
DATA_FINAL = "31/12/2023"
URL_SGS = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"

In [3]:
def extrair_serie_sgs(codigo, data_inicial, data_final, tentativas=3):
    """Baixa uma série do SGS/BACEN e devolve um DataFrame com as colunas 'data' e 'valor'."""
    params = {"formato": "json", "dataInicial": data_inicial, "dataFinal": data_final}
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(
                URL_SGS.format(codigo=codigo), params=params, timeout=30
            )
            resposta.raise_for_status()
            return pd.DataFrame(resposta.json())
        except requests.RequestException as erro:
            print(f"  Série {codigo}: tentativa {tentativa} falhou ({erro})")
            time.sleep(2 * tentativa)
    raise RuntimeError(f"Não foi possível baixar a série {codigo} do SGS.")

In [4]:
resumo = []
for codigo, nome in SERIES_SGS.items():
    df = extrair_serie_sgs(codigo, DATA_INICIAL, DATA_FINAL)
    df.to_csv(RAW_DIR / f"bacen_{codigo}_{nome}.csv", index=False, encoding="utf-8")
    resumo.append(
        {
            "codigo": codigo,
            "serie": nome,
            "linhas": len(df),
            "primeira_data": df["data"].iloc[0],
            "ultima_data": df["data"].iloc[-1],
        }
    )
    time.sleep(1)  # pausa entre requisições: respeita os limites da API

pd.DataFrame(resumo)

,codigo,serie,linhas,primeira_data,ultima_data
0,432,selic_meta,1826,01/01/2019,31/12/2023
1,433,ipca_mensal,60,01/01/2019,01/12/2023
2,13522,ipca_12m,60,01/01/2019,01/12/2023
3,1,ptax_venda,1255,02/01/2019,29/12/2023


In [5]:
selic_bruta = pd.read_csv(RAW_DIR / "bacen_432_selic_meta.csv")
print(selic_bruta.dtypes)
selic_bruta.head()

data         str
valor    float64
dtype: object


,data,valor
0,01/01/2019,6.5
1,02/01/2019,6.5
2,03/01/2019,6.5
3,04/01/2019,6.5
4,05/01/2019,6.5


## Conferência da extração

| Série | Linhas | Validação |
|---|---|---|
| 432 — Selic Meta | 1.826 | Série diária corrida: 365 + 366 + 365 + 365 + 365 dias (2020 bissexto) |
| 433 — IPCA mensal | 60 | 5 anos × 12 meses |
| 13522 — IPCA 12m | 60 | 5 anos × 12 meses |
| 1 — PTAX venda | 1.255 | Apenas dias úteis (~250/ano); início em 02/01/2019 (feriado em 01/01) |

**Observação para a etapa de limpeza:** a coluna `data` está como texto no formato brasileiro (dd/mm/aaaa) e deve ser convertida com formato explícito (`%d/%m/%Y`), evitando a leitura no padrão americano (mm/dd). A coluna `valor` já é numérica (`float64`).